In [25]:
import gymnasium as gym
import numpy as np

In [26]:
def build_env():
    env = gym.make("gymnasium_2048:gymnasium_2048/TwentyFortyEight-v0", size=4, max_pow=16)
    env.reset()
    return env

In [27]:
def features(state,action):
    feature = state.flatten()
    possible_actions = [[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]]
    feature = np.append(feature,possible_actions[action])
    return feature

In [28]:
def q_hat(state, action, weights):
    return np.dot(weights.transpose(), features(state,action))

In [29]:
def q_hat_grad(state,action):
    return features(state,action)

In [30]:
def sarsa(env,episodes,learning_rate,gamma,epsilon,max_steps):
    weights = np.zeros(260)
    for _ in range(episodes):
        state,_ = env.reset()
        possibleactions = [0,1,2,3]
        if np.random.rand() < epsilon:
            action = np.random.choice(possibleactions)
        else :
            action = np.argmax([q_hat(state,actions,weights) for actions in possibleactions])
        done = False
        for _ in range(max_steps):
            next_state, reward, done, _, _ = env.step(action)
            if np.random.rand()<epsilon:
                next_action = np.random.choice(possibleactions)
            else:
                next_action = np.argmax([q_hat(next_state,actions,weights) for actions in possibleactions])
            if done == True:
                weights +=learning_rate*(reward - q_hat(next_state,next_action,weights))*(q_hat_grad(next_state,next_action))
                break
            weights +=learning_rate*(reward + gamma*q_hat(next_state,next_action,weights)-q_hat(state,action,weights))*(q_hat_grad(state,action))
            state = next_state
            action = next_action
    return weights

In [31]:
def main():
    gamma = 0.99
    learning_rate = 0.003
    epsilon = 0.1
    episodes = 100
    max_steps = 2000
    env = build_env()
    weights = sarsa(env, episodes, learning_rate, gamma, epsilon, max_steps)
    print(weights)
    total_reward = 0  
    for i in range(100):
        state, _ = env.reset()
        maxr = 0
        for _ in range(max_steps):
            q_values = [q_hat(state, a, weights) for a in range(4)]
            if np.random.rand() < 0.05:
                action = np.random.randint(0,4)
            else :
                action = np.argmax(q_values)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                print(f"Episode ended.{total_reward} , {i}")
                break
    print(f"Test episode total reward: {total_reward/100}")

In [32]:
if __name__ == "__main__":
    main()

[ 1.73183919e+01  1.74243898e+01  1.55054109e+01  8.84657014e+00
  3.81210160e+00 -1.11895534e+00  3.07362466e-01  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.48009073e+01  1.44499083e+01  1.07352004e+01  1.14886227e+01
  4.98223963e+00  5.37463955e+00  8.13778909e-02  1.82375602e-01
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.34743966e+01  1.40060182e+01  8.94116326e+00  9.32178836e+00
  9.42253663e+00  2.50247718e+00  5.22395047e+00 -7.97059148e-01
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.50866517e+01  1.47478686e+01  1.20325252e+01  8.64750468e+00
  6.86090345e+00  2.15267269e+00  2.56714515e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000